In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import japanize_matplotlib # 日本語表示に対応


import datetime
import pymysql
import numpy as np
import python_ss.python_ss as ps
import gspread
from oauth2client.service_account import ServiceAccountCredentials
import json

import os
import ast
import db_dtypes
from google.cloud import bigquery
from google.oauth2 import service_account
from google.cloud import secretmanager
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build
from google_auth_oauthlib.flow import InstalledAppFlow
# importでエラーが出てしまった場合は、コマンドプロンプトにて「pip install ”必要なモジュール”」でインストールしていただく必要がございます。
# 例. pip install db_dtypes

import xlsxwriter

from decimal import Decimal
import calendar
# import utils
# from utils import *
#importlib.reload(utils)
print(os.getcwd())



z:\Users\suehara\Documents\python\analysis


In [ ]:


# --- 1. データの読み込みと前処理 ---
print("--- データの読み込みと基本情報の確認 ---")
# CSVファイルを読み込みます
df = pd.read_csv('ronzan_data.csv')

# 日付形式の列をdatetime型に変換します
date_columns = ['shoki_setteibi', 'shoki_jisshibi', 'hon_setteibi', 'hon_yoteibi', 'hon_jisshibi', 'seiyakubi']
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce') # errors='coerce'で変換できない値はNaT(欠損)にする

# 予測のターゲットとなる「成約フラグ」を作成します (成約日があれば1, なければ0)
df['seiyaku_flag'] = df['seiyakubi'].notna().astype(int)

# データの基本情報を表示します
print("▼ データ概要 (info)")
df.info()

print("\n▼ 数値データの統計量 (describe)")
print(df.describe())

print("\n▼ データの一部を表示 (head)")
print(df.head())


# --- 2. データの可視化 ---
print("\n--- データの可視化を実行中 ---")

# グラフのスタイル設定
sns.set(style='whitegrid')

# ① APsource別の案件数
plt.figure(figsize=(12, 6))
sns.countplot(data=df, x='APsource', order=df['APsource'].value_counts().index)
plt.title('グラフ1: APsource別の案件数')
plt.xticks(rotation=45)
plt.tight_layout() # ラベルが重ならないように調整
plt.show()

# ② 成約・非成約の案件数
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='seiyaku_flag')
plt.title('グラフ2: 成約・非成約の案件数')
plt.xticks([0, 1], ['非成約', '成約'])
plt.show()

# ③ 成約リードタイムの分布 (ヒストグラム)
# 成約した案件のみを対象
contracted_df = df.dropna(subset=['seiyaku_lead_time'])
plt.figure(figsize=(10, 6))
sns.histplot(data=contracted_df, x='seiyaku_lead_time', bins=30, kde=True)
plt.title('グラフ3: 成約リードタイムの分布（日数）')
plt.xlabel('本交渉実施から成約までの日数')
plt.ylabel('案件数')
plt.show()

# ④ APsource別の成約率
# APsourceごとの成約率を計算
apsource_seiyaku_rate = df.groupby('APsource')['seiyaku_flag'].mean().sort_values(ascending=False)

plt.figure(figsize=(12, 6))
apsource_seiyaku_rate.plot(kind='bar')
plt.title('グラフ4: APsource別の成約率')
plt.ylabel('成約率')
plt.xticks(rotation=45)
plt.grid(axis='y')
plt.tight_layout()
plt.show()

print("\n--- 可視化完了 ---")

In [5]:
def access_secret_version(project_id, secret_id, version_id='latest'):
    client = secretmanager.SecretManagerServiceClient()

    name = f"projects/{project_id}/secrets/{secret_id}/versions/{version_id}"
    response = client.access_secret_version(request={"name": name})
    payload = response.payload.data.decode("UTF-8")
    return ast.literal_eval(payload)

In [6]:
# 上記関数を実行するコードが記載されています。こちらもそのままお使いください。
credentials = service_account.Credentials.from_service_account_info(
access_secret_version('temp-for-sandbox', 'TEMP_CREDENTIAL_KEY'),
scopes=["https://www.googleapis.com/auth/cloud-platform"],)


NameError: name 'service_account' is not defined

In [4]:
# 上記関数を実行するコードが記載されています。こちらもそのままお使いください。
credentials = service_account.Credentials.from_service_account_info(
access_secret_version('r-group-bigdata', 'CREDENTIALS_SECRET_KEY_WORKER'),
scopes=["https://www.googleapis.com/auth/cloud-platform"],)


In [5]:
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

## 各種データ取得

In [6]:
qry = """
WITH shoki AS (
  SELECT
    kohosha_id,
    MAX(kosho_jisshibi) AS last_mendan_date,
    kohosha_biko
  FROM
    `r-group-bigdata.live_rhs.shokikoshos`
  GROUP BY
    kohosha_id,
    kohosha_biko
),

 koho as (SELECT 
 id,
 shusshin_kigyo,
 max_bushoyakushoku_str,
 birth_year,
 annual_income,
CASE WHEN koho.pref = 3091 THEN "北海道"
     WHEN koho.pref = 3102 THEN "青森県"
     WHEN koho.pref = 3103 THEN "岩手県"
     WHEN koho.pref = 3104 THEN "宮城県"
     WHEN koho.pref = 3105 THEN "秋田県"
     WHEN koho.pref = 3106 THEN "山形県"
     WHEN koho.pref = 3107 THEN "福島県"
     WHEN koho.pref = 3128 THEN "茨城県"
     WHEN koho.pref = 3129 THEN "栃木県"
     WHEN koho.pref = 31210 THEN "群馬県"
     WHEN koho.pref = 31211 THEN "埼玉県"
     WHEN koho.pref = 31212 THEN "千葉県"
     WHEN koho.pref = 31213 THEN "東京都"
     WHEN koho.pref = 31214 THEN "神奈川県"
     WHEN koho.pref = 31115 THEN "新潟県"
     WHEN koho.pref = 31116 THEN "富山県"
     WHEN koho.pref = 31117 THEN "石川県"
     WHEN koho.pref = 31118 THEN "福井県"
     WHEN koho.pref = 31319 THEN "山梨県"
     WHEN koho.pref = 31320 THEN "長野県"
     WHEN koho.pref = 31321 THEN "岐阜県"
     WHEN koho.pref = 31322 THEN "静岡県"
     WHEN koho.pref = 31323 THEN "愛知県"
     WHEN koho.pref = 31324 THEN "三重県"
     WHEN koho.pref = 31425 THEN "滋賀県"
     WHEN koho.pref = 31426 THEN "京都府"
     WHEN koho.pref = 31427 THEN "大阪府"
     WHEN koho.pref = 31428 THEN "兵庫県"
     WHEN koho.pref = 31429 THEN "奈良県"
     WHEN koho.pref = 31430 THEN "和歌山県"
     WHEN koho.pref = 31531 THEN "鳥取県"
     WHEN koho.pref = 31532 THEN "島根県"
     WHEN koho.pref = 31533 THEN "岡山県"
     WHEN koho.pref = 31534 THEN "広島県"
     WHEN koho.pref = 31535 THEN "山口県"
     WHEN koho.pref = 31636 THEN "徳島県"
     WHEN koho.pref = 31637 THEN "香川県"
     WHEN koho.pref = 31638 THEN "愛媛県"
     WHEN koho.pref = 31639 THEN "高知県"
     WHEN koho.pref = 31740 THEN "福岡県"
     WHEN koho.pref = 31741 THEN "佐賀県"
     WHEN koho.pref = 31742 THEN "長崎県"
     WHEN koho.pref = 31743 THEN "熊本県"
     WHEN koho.pref = 31744 THEN "大分県"
     WHEN koho.pref = 31745 THEN "宮崎県"
     WHEN koho.pref = 31746 THEN "鹿児島県"
     WHEN koho.pref = 31847 THEN "沖縄県"
     WHEN koho.pref = 90000 THEN "海外"
ELSE "不明" END AS pref
FROM `r-group-bigdata.live_rhs.kohoshas` koho
where kohosha_status = 10)

SELECT distinct
  file.kohosha_id,
  shoki.last_mendan_date,
  shusshin_kigyo,
  EXTRACT(YEAR FROM CURRENT_DATE()) - koho.birth_year AS age,
  max_bushoyakushoku_str,
  annual_income,
  pref,  
  file.texts,             
  shoki.kohosha_biko,
FROM `r-group-bigdata.live_rhs.kohosha_files` AS file 
left join shoki on file.kohosha_id = shoki.kohosha_id
left join koho on file.kohosha_id = koho.id
WHERE shoki.last_mendan_date >= DATE_SUB(CURRENT_DATE(), INTERVAL 1 YEAR)
  AND shoki.last_mendan_date IS NOT NULL
  and pref IN ('東京都', '神奈川県', '千葉県', '埼玉県')
-- limit 10
"""

client = bigquery.Client(credentials=credentials, project=credentials.project_id)
raw_data = client.query(qry).result().to_dataframe()